In [ ]:
import os
import random
import json
import cv2
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from sklearn.metrics import f1_score
from tqdm import tqdm
from ensemble_boxes import weighted_boxes_fusion
from ultralytics import YOLO
from PIL import Image

# -------------------- НАСТРОЙКИ --------------------
DATA_ROOT = "D:/DL_2"
TRAIN_IMAGES_DIR = "D:/DL_2/yolo_dataset/yolo_dataset/train/images"
TRAIN_LABELS_DIR = "D:/DL_2/yolo_dataset/yolo_dataset/train/labels"
TEST_IMAGES_DIR = "D:/DL_2/test_images/test_images"
SAMPLE_SUB = f"{DATA_ROOT}/sample_sub.csv"

SPLIT_DIR = "D:/DL_2/datasets_split"
TRAIN_SPLIT = f"{SPLIT_DIR}/train"
VAL_SPLIT = f"{SPLIT_DIR}/val"

YOLO_WEIGHTS = [
    "D:/DL_2/yolo_models/yolov8m/weights/best.pt",
    "D:/DL_2/yolo_models/yolov9m/weights/best.pt",
    "D:/DL_2/yolo_models/yolov10m/weights/best.pt"
]

CLASSIFIER_INPUT_SIZE = 256
CLASSIFIER_BATCH_SIZE = 32
CLASSIFIER_EPOCHS = 50
CLASSIFIER_LR = 0.001
CLASSIFIER_WEIGHT_DECAY = 1e-4

NUM_CLASSES = 2
CLASS_NAMES = ['customer', 'staff']

CONF_THRESHOLD = 0.15
FINAL_CONF_THRESHOLD = 0.35
WBF_IOU_THR = 0.5

OUTPUT_CSV = "submission_with_classes.csv"
# ---------------------------------------------------

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Создаем папки
for dir_path in [SPLIT_DIR, TRAIN_SPLIT, VAL_SPLIT]:
    os.makedirs(f"{dir_path}/images", exist_ok=True)
    os.makedirs(f"{dir_path}/labels", exist_ok=True)

def split_data_stratified():
    """Стратифицированное разделение данных"""
    images = [f for f in os.listdir(TRAIN_IMAGES_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))]
    
    bins = defaultdict(list)
    for img in images:
        label_path = os.path.join(TRAIN_LABELS_DIR, img.replace('.jpg', '.txt').replace('.png', '.txt'))
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                num_objects = sum(1 for line in f if line.strip())
        else:
            num_objects = 0
        
        if num_objects == 0:
            bin_key = '0'
        elif num_objects <= 2:
            bin_key = '1-2'
        elif num_objects <= 5:
            bin_key = '3-5'
        else:
            bin_key = '6+'
        bins[bin_key].append(img)
    
    train_imgs, val_imgs = [], []
    for bin_key, bin_images in bins.items():
        random.seed(42)
        random.shuffle(bin_images)
        split_idx = int(0.85 * len(bin_images))
        train_imgs.extend(bin_images[:split_idx])
        val_imgs.extend(bin_images[split_idx:])
        
        # Копируем файлы
        for img in train_imgs:
            shutil.copy(f"{TRAIN_IMAGES_DIR}/{img}", f"{TRAIN_SPLIT}/images/{img}")
            label = img.replace('.jpg', '.txt').replace('.png', '.txt').replace('.jpeg', '.txt')
            if os.path.exists(f"{TRAIN_LABELS_DIR}/{label}"):
                shutil.copy(f"{TRAIN_LABELS_DIR}/{label}", f"{TRAIN_SPLIT}/labels/{label}")
            else:
                open(f"{TRAIN_SPLIT}/labels/{label}", 'w').close()
        
        for img in val_imgs:
            shutil.copy(f"{TRAIN_IMAGES_DIR}/{img}", f"{VAL_SPLIT}/images/{img}")
            label = img.replace('.jpg', '.txt').replace('.png', '.txt').replace('.jpeg', '.txt')
            if os.path.exists(f"{TRAIN_LABELS_DIR}/{label}"):
                shutil.copy(f"{TRAIN_LABELS_DIR}/{label}", f"{VAL_SPLIT}/labels/{label}")
            else:
                open(f"{VAL_SPLIT}/labels/{label}", 'w').close()
    
    print(f"\nFinal split: Train: {len(train_imgs)} images, Val: {len(val_imgs)} images")
    return train_imgs, val_imgs

class PatchDataset(Dataset):
    """Датасет для патчей с людьми"""
    def __init__(self, image_dir, label_dir, is_train=True):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.is_train = is_train
        self.image_files = [f for f in os.listdir(image_dir) if f.endswith(('.jpg', '.png', '.jpeg'))]
        
        if is_train:
            self.transform = transforms.Compose([
                transforms.RandomResizedCrop(CLASSIFIER_INPUT_SIZE, scale=(0.7, 1.0)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomRotation(15),
                transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transforms.Compose([
                transforms.Resize((CLASSIFIER_INPUT_SIZE, CLASSIFIER_INPUT_SIZE)),
                transforms.ToTensor(),
                transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
            ])
        
        # Загружаем метки
        self.labels_cache = {}
        for img_name in self.image_files:
            label_name = Path(img_name).with_suffix('.txt').name
            label_path = os.path.join(label_dir, label_name)
            boxes = []
            if os.path.exists(label_path):
                with open(label_path, 'r') as f:
                    for line in f:
                        parts = line.strip().split()
                        if len(parts) == 5:
                            class_id = int(parts[0])
                            xc, yc, w, h = map(float, parts[1:5])
                            boxes.append([class_id, xc, yc, w, h])
            self.labels_cache[img_name] = boxes
        
        # Собираем патчи
        self.patches_data = []
        self._prepare_patches()
    
    def _prepare_patches(self):
        """Собираем патчи с людьми"""
        for img_name in tqdm(self.image_files, desc="Preparing patches"):
            img_path = os.path.join(self.image_dir, img_name)
            image = cv2.imread(img_path)
            if image is None: continue
            
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            h, w, _ = image.shape
            boxes = self.labels_cache.get(img_name, [])
            
            for box in boxes:
                class_id, xc, yc, bw, bh = box
                if class_id in [0, 1]:
                    x1 = int((xc - bw/2) * w)
                    y1 = int((yc - bh/2) * h)
                    x2 = int((xc + bw/2) * w)
                    y2 = int((yc + bh/2) * h)
                    
                    # Padding 10%
                    pad = int(0.1 * max(x2 - x1, y2 - y1))
                    x1 = max(0, x1 - pad)
                    y1 = max(0, y1 - pad)
                    x2 = min(w, x2 + pad)
                    y2 = min(h, y2 + pad)
                    
                    if (x2 - x1) > 10 and (y2 - y1) > 10:
                        self.patches_data.append((img_name, [x1, y1, x2, y2], class_id))
        
        print(f"\nPrepared {len(self.patches_data)} patches:")
        print(f"  Customers (0): {sum(1 for _,_,l in self.patches_data if l==0)}")
        print(f"  Staff (1): {sum(1 for _,_,l in self.patches_data if l==1)}")
    
    def __len__(self):
        return len(self.patches_data)
    
    def __getitem__(self, idx):
        img_name, bbox, label = self.patches_data[idx]
        img_path = os.path.join(self.image_dir, img_name)
        
        # Чтение изображения
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        x1, y1, x2, y2 = bbox
        patch = image[y1:y2, x1:x2]
        
        # Защита от пустых патчей
        if patch.size == 0 or patch.shape[0] == 0 or patch.shape[1] == 0:
            patch = np.zeros((CLASSIFIER_INPUT_SIZE, CLASSIFIER_INPUT_SIZE, 3), dtype=np.uint8)
        
        # Конвертируем в PIL Image
        patch_pil = Image.fromarray(patch)
        
        if self.transform:
            patch = self.transform(patch_pil)
        
        return patch, label

class EfficientNetClassifier(nn.Module):
    def __init__(self, num_classes=2, dropout_rate=0.5):
        super().__init__()
        self.backbone = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
        
        # Замораживаем первые слои
        for i, param in enumerate(self.backbone.features.parameters()):
            if i < 30:
                param.requires_grad = False
        
        # Получаем количество каналов
        in_features = self.backbone.classifier[1].in_features
        
        # Заменяем классификатор
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=dropout_rate, inplace=True),
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate * 0.7),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout_rate * 0.5),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        return self.backbone(x)

def train_classifier():
    """Обучение классификатора"""
    print("\n" + "="*60)
    print("TRAINING CLASSIFIER")
    print("="*60)
    
    print("\nCreating datasets...")
    train_dataset = PatchDataset(TRAIN_SPLIT + "/images", TRAIN_SPLIT + "/labels", is_train=True)
    val_dataset = PatchDataset(VAL_SPLIT + "/images", VAL_SPLIT + "/labels", is_train=False)
    
    train_loader = DataLoader(
        train_dataset, 
        batch_size=CLASSIFIER_BATCH_SIZE, 
        shuffle=True, 
        num_workers=0,
        pin_memory=True
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=CLASSIFIER_BATCH_SIZE * 2, 
        shuffle=False, 
        num_workers=0,
        pin_memory=True
    )
    
    model = EfficientNetClassifier(num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=CLASSIFIER_LR, weight_decay=CLASSIFIER_WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)
    
    best_acc = 0.0
    patience_counter = 0
    max_patience = 10
    
    for epoch in range(CLASSIFIER_EPOCHS):
        # Training
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{CLASSIFIER_EPOCHS} [Train]")
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = outputs.max(1)
            train_total += labels.size(0)
            train_correct += predicted.eq(labels).sum().item()
            
            pbar.set_postfix({
                'loss': f"{train_loss/(train_total/CLASSIFIER_BATCH_SIZE+1):.4f}",
                'acc': f"{100.*train_correct/train_total:.2f}%"
            })
        
        # Validation
        model.eval()
        val_correct = 0
        val_total = 0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{CLASSIFIER_EPOCHS} [Val]"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
                
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        val_acc = 100. * val_correct / val_total
        val_f1 = f1_score(all_labels, all_preds, average='weighted')
        
        print(f"\nEpoch {epoch+1} Results:")
        print(f"  Train Acc: {100.*train_correct/train_total:.2f}%")
        print(f"  Val Acc: {val_acc:.2f}%, Val F1: {val_f1:.4f}")
        
        # Save best model
        if val_acc > best_acc:
            best_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'val_acc': val_acc,
                'val_f1': val_f1,
            }, 'best_classifier.pth')
            print(f"  -> Saved new best model with Acc: {val_acc:.2f}%")
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= max_patience:
                print(f"Early stopping after {epoch+1} epochs")
                break
        
        scheduler.step()
    
    # Load best model
    checkpoint = torch.load('best_classifier.pth')
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"\nLoaded best model with Acc: {checkpoint['val_acc']:.2f}%, F1: {checkpoint['val_f1']:.4f}")
    
    return model

class InferenceEngine:
    def __init__(self, yolo_models, classifier, device):
        self.yolo_models = yolo_models
        self.classifier = classifier
        self.device = device
        
        self.transform = transforms.Compose([
            transforms.Resize((CLASSIFIER_INPUT_SIZE, CLASSIFIER_INPUT_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
    
    def predict_batch(self, image_paths, batch_size=8):
        all_results = []
        
        for i in range(0, len(image_paths), batch_size):
            batch_paths = image_paths[i:i+batch_size]
            batch_results = []
            
            # YOLO предсказания
            all_yolo_boxes = [[] for _ in range(len(self.yolo_models))]
            all_yolo_scores = [[] for _ in range(len(self.yolo_models))]
            all_yolo_labels = [[] for _ in range(len(self.yolo_models))]
            
            for img_path in batch_paths:
                for idx, model in enumerate(self.yolo_models):
                    results = model(img_path, verbose=False, conf=CONF_THRESHOLD)[0]
                    if len(results.boxes) > 0:
                        boxes = results.boxes.xyxyn.cpu().numpy()
                        scores = results.boxes.conf.cpu().numpy()
                        all_yolo_boxes[idx].append(boxes.tolist())
                        all_yolo_scores[idx].append(scores.tolist())
                        all_yolo_labels[idx].append([0] * len(boxes))
                    else:
                        all_yolo_boxes[idx].append([])
                        all_yolo_scores[idx].append([])
                        all_yolo_labels[idx].append([])
            
            # WBF fusion
            for j in range(len(batch_paths)):
                boxes_list = [all_yolo_boxes[m][j] for m in range(len(self.yolo_models))]
                scores_list = [all_yolo_scores[m][j] for m in range(len(self.yolo_models))]
                labels_list = [all_yolo_labels[m][j] for m in range(len(self.yolo_models))]
                
                if any(len(b) > 0 for b in boxes_list):
                    fused_boxes, fused_scores, _ = weighted_boxes_fusion(
                        boxes_list, scores_list, labels_list,
                        iou_thr=WBF_IOU_THR,
                        skip_box_thr=0.0001,
                    )
                    batch_results.append((fused_boxes, fused_scores))
                else:
                    batch_results.append(([], []))
            
            # Классификация патчей
            all_patches = []
            all_indices = []
            
            for j, (fused_boxes, fused_scores) in enumerate(batch_results):
                if len(fused_boxes) == 0:
                    continue
                
                img_path = batch_paths[j]
                image = cv2.imread(img_path)
                if image is None: continue
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                h, w, _ = image.shape
                
                for box_idx, box in enumerate(fused_boxes):
                    x1, y1, x2, y2 = box
                    abs_x1 = int(np.clip(x1 * w, 0, w - 1))
                    abs_y1 = int(np.clip(y1 * h, 0, h - 1))
                    abs_x2 = int(np.clip(x2 * w, 0, w - 1))
                    abs_y2 = int(np.clip(y2 * h, 0, h - 1))
                    
                    patch = image[abs_y1:abs_y2, abs_x1:abs_x2]
                    
                    if patch.size > 0 and patch.shape[0] > 1 and patch.shape[1] > 1:
                        patch_pil = Image.fromarray(patch)
                        patch_tensor = self.transform(patch_pil)
                        all_patches.append(patch_tensor)
                        all_indices.append((j, box, fused_scores[box_idx]))
            
            # Классификация
            current_batch_results = [[] for _ in range(len(batch_paths))]
            
            if all_patches:
                patches_batch = torch.stack(all_patches).to(self.device)
                self.classifier.eval()
                with torch.no_grad():
                    outputs = self.classifier(patches_batch)
                    probabilities = torch.softmax(outputs, dim=1)
                    predicted_classes = torch.argmax(probabilities, dim=1).cpu().numpy()
                    confidences = probabilities[range(len(predicted_classes)), predicted_classes].cpu().numpy()
                
                for (j, box, yolo_score), pred_class, conf in zip(all_indices, predicted_classes, confidences):
                    if conf >= FINAL_CONF_THRESHOLD:
                        x1, y1, x2, y2 = box
                        xc = (x1 + x2) / 2
                        yc = (y1 + y2) / 2
                        wb = x2 - x1
                        hb = y2 - y1
                        current_batch_results[j].append([int(pred_class), float(xc), float(yc), float(wb), float(hb), float(conf)])
            
            all_results.extend(current_batch_results)
        
        return all_results

def main():
    import shutil
    
    # Шаг 1: Разделение данных (если нужно)
    print("\nStep 1: Checking data split...")
    if not os.path.exists(TRAIN_SPLIT) or len(os.listdir(f"{TRAIN_SPLIT}/images")) == 0:
        train_imgs, val_imgs = split_data_stratified()
    else:
        print("Split directory already exists, skipping...")
    
    # Шаг 2: Обучение классификатора
    classifier = train_classifier()
    classifier.eval()
    
    # Шаг 3: Загрузка YOLO моделей
    print("\nStep 3: Loading YOLO models...")
    yolo_models = []
    for weight_path in YOLO_WEIGHTS:
        if os.path.exists(weight_path):
            print(f"Loading: {weight_path}")
            yolo_models.append(YOLO(weight_path))
        else:
            print(f"WARNING: {weight_path} not found!")
    
    print(f"Loaded {len(yolo_models)} YOLO models")
    
    # Шаг 4: Инференс
    print("\nStep 4: Running inference...")
    sample_df = pd.read_csv(SAMPLE_SUB)
    image_names = sample_df['image_name'].tolist()
    image_paths = [os.path.join(TEST_IMAGES_DIR, img_name) for img_name in image_names]
    
    inference_engine = InferenceEngine(yolo_models, classifier, device)
    
    print(f"\nProcessing {len(image_paths)} test images...")
    all_predictions = inference_engine.predict_batch(image_paths, batch_size=CLASSIFIER_BATCH_SIZE)
    
    # Проверка
    if len(all_predictions) != len(image_names):
        print(f"WARNING: Predictions count mismatch! Got {len(all_predictions)}, expected {len(image_names)}")
        while len(all_predictions) < len(image_names):
            all_predictions.append([])
    
    # Шаг 5: Формирование submission
    print("\nStep 5: Creating submission file...")
    results_rows = []
    for idx, (img_name, boxes) in enumerate(zip(image_names, all_predictions)):
        results_rows.append({
            'id': idx,
            'image_name': img_name,
            'boxes': json.dumps(boxes, separators=(',', ':'))
        })
    
    sub_df = pd.DataFrame(results_rows, columns=['id', 'image_name', 'boxes'])
    sub_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Submission saved to: {OUTPUT_CSV}")
    
    # Шаг 6: Статистика
    print("\nStep 6: Prediction statistics:")
    total_boxes = 0
    images_with_boxes = 0
    customers = 0
    staff = 0
    
    for boxes in all_predictions:
        if boxes:
            images_with_boxes += 1
            total_boxes += len(boxes)
            for box in boxes:
                if box[0] == 0:
                    customers += 1
                else:
                    staff += 1
    
    print(f"Total test images: {len(image_names)}")
    print(f"Images with detections: {images_with_boxes} ({100.*images_with_boxes/len(image_names):.1f}%)")
    print(f"Total bounding boxes: {total_boxes}")
    print(f"  - Customers: {customers}")
    print(f"  - Staff: {staff}")
    print(f"Average boxes per image: {total_boxes/len(image_names):.2f}")
    
    print("\n" + "="*50)
    print("DONE!")
    print("="*50)

if __name__ == "__main__":
    main()

Using device: cuda

Step 1: Checking data split...
Split directory already exists, skipping...

TRAINING CLASSIFIER

Creating datasets...


Preparing patches: 100%|██████████| 3320/3320 [00:17<00:00, 184.59it/s]



Prepared 17907 patches:
  Customers (0): 14441
  Staff (1): 3466


Preparing patches: 100%|██████████| 588/588 [00:03<00:00, 185.86it/s]



Prepared 3191 patches:
  Customers (0): 2598
  Staff (1): 593


...

Epoch 18/50 [Val]: 100%|██████████| 50/50 [00:27<00:00,  1.84it/s]



Epoch 18 Results:
  Train Acc: 98.71%
  Val Acc: 97.87%, Val F1: 0.9789


Epoch 19/50 [Val]: 100%|██████████| 50/50 [00:40<00:00,  1.23it/s]



Epoch 19 Results:
  Train Acc: 98.89%
  Val Acc: 98.18%, Val F1: 0.9819
Early stopping after 19 epochs

Loaded best model with Acc: 98.68%, F1: 0.9869

Step 3: Loading YOLO models...
Loading: D:/DL_2/yolo_models/yolov8m/weights/best.pt
Loading: D:/DL_2/yolo_models/yolov9m/weights/best.pt
Loading: D:/DL_2/yolo_models/yolov10m/weights/best.pt
Loaded 3 YOLO models

Step 4: Running inference...

Processing 4454 test images...


FileNotFoundError: D:/DL_2/test_images\32_09-02-30-000.jpg does not exist